# Notebook for the LLM processing of medical reports

In [1]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

The data contains text documents that are annotated for mentions of participants, interventions and outcomes (PIO) in medical research. For each entity type, P, I, or O, there is a slightly different set of documents in the training and test set. Most of the documents are identical, but each type has a few extra documents. Looking at the lecture notes and doing some research online it seems clear that the best approach currently available for the task of token classification are transformer models, specifically transformer encoders, the most famous example being the BERT (Bidirectional Encoder Representations from Transformers) model developed at Google. In order to test these models a group called Huggingfaces has created a website where people can upload their deep learning models and they have many examples of different BERT models that have been developed as well as providing a tutorial on how to use the API they have developed for quickly loading and utilising these models. The model I have looked at is one of the smallest versions: DistilBERT

However, first we need to load the data from the files so here I used the code provided.    
To load the text documents, we first make a list of the document IDs for one entity type (P, I or O):

In [2]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_2_00")

docs_dir = DATA_DIR / "documents"

def get_doc_ids(split="train", label_type="interventions"):
    """ 
    split: 'train' or 'test' 
    """

    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type  # assuming that the split is the same for all entity types, we can just look at one of them
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
   # print(doc_ids)
    return sorted(doc_ids)

doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

print(f"Number of documents in train split for interventions: {len(doc_ids_i)}")
print(f"Number of documents in test split for interventions: {len(test_doc_ids_i)}")

Number of documents in train split for interventions: 4746
Number of documents in test split for interventions: 187


In [3]:
def load_labels_for_doc(doc_id, label_type="interventions", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test' 
    """
    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="interventions", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

interventions_labels = load_labels(doc_ids_i, "interventions", split="train")

print(f"Length of participants_labels: {len(interventions_labels)}")

test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")
print(f"Length of test_participants_labels: {len(test_interventions_labels)}")

sample = 123
print("Document ID:", doc_ids_i[sample])
print(f"Interventions label example for doc {doc_ids_i[sample]}:")
print(interventions_labels[sample])

Length of participants_labels: 4746
Length of test_participants_labels: 187
Document ID: 10674680
Interventions label example for doc 10674680:
['0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', 

In [4]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

# inspect a random element
print("Document ID:", doc_ids_i[sample])
print(f"Tokenised document example for doc {doc_ids_i[sample]}:")
print(interventions_tokens[sample])
print(interventions_labels[sample])

Document ID: 10674680
Tokenised document example for doc 10674680:
['Assessment', 'of', 'therapeutic', 'response', 'of', 'Plasmodium', 'falciparum', 'to', 'chloroquine', 'and', 'sulfadoxine-pyrimethamine', 'in', 'an', 'area', 'of', 'low', 'malaria', 'transmission', 'in', 'Colombia', '.', 'Although', 'chloroquine', '(', 'CQ', ')', 'resistance', 'was', 'first', 'reported', 'in', 'Colombia', 'in', '1961', 'and', 'sulfadoxine-pyrimethamine', '(', 'SP', ')', 'resistance', 'in', '1981', ',', 'the', 'frequency', 'of', 'treatment', 'failures', 'to', 'these', 'drugs', 'in', 'Colombia', 'is', 'unclear', '.', 'A', 'modified', 'World', 'Health', 'Organization', '14-day', 'in', 'vivo', 'drug', 'efficacy', 'test', 'for', 'uncomplicated', 'Plasmodium', 'falciparum', 'malaria', 'in', 'areas', 'with', 'intense', 'malaria', 'transmission', 'was', 'adapted', 'to', 'reflect', 'the', 'clinical', 'and', 'epidemiologic', 'features', 'of', 'a', 'low-intensity', 'malaria', 'transmission', 'area', 'in', 'the', 

In [5]:
def load_text_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.txt"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

text = load_text_document(10674680)

text

['Assessment of therapeutic response of Plasmodium falciparum to chloroquine and sulfadoxine-pyrimethamine in an area of low malaria transmission in Colombia.',
 '',
 'Although chloroquine (CQ) resistance was first reported in Colombia in 1961 and sulfadoxine-pyrimethamine (SP) resistance in 1981, the frequency of treatment failures to these drugs in Colombia is unclear. A modified World Health Organization 14-day in vivo drug efficacy test for uncomplicated Plasmodium falciparum malaria in areas with intense malaria transmission was adapted to reflect the clinical and epidemiologic features of a low-intensity malaria transmission area in the Pacific Coast Region of Colombia. Patients > or =1 year of age with a parasite density > or =1,000 asexual parasites per microliter were enrolled in this study. Forty-four percent (24 of 54) of the CQ-treated patients were therapeutic failures, including 7 early treatment failures (ETFs) and 17 late treatment failures (LTFs). Four (6%) of 67 SP-tr

I decided to start the processing with identifying interventions. This is because I thought that this would be the simplest one as most of the interventions are named drugs so while each drug might be unique, the contexts in which they appear should all be similar.

### Process to follow:
1. Load the files
2. Convert labels to ones combining the B I O labels with the ner tags
3. Convert combined labels to numeric ids
4. Initialize the model and tokenizer
5. Tokenize the words and align the labels with any words that have been split into multiple tokens
6. Prepare the model
7. Create the compute metrics function
8. Add the training arguments
9. Run the model
10. Repeat for Participants and Outcomes

In [6]:
from itertools import chain
import numpy as np

all_labels = chain(*interventions_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['0' '1' '2' '3' '4' '5' '6' '7']


### Reformatting the labels

In order to do token classification it helps to label the starting points of tokens (typically noted as B) as well as points that are "inside" the token (i.e. the second part of a name) as this helps split tokens that are next to each other. As our data is labelled 0-7 as seen above we can alter the labels by adding "B-" or "I-" to the start of each one (except for the zeros which are replaced with the letter O). Therefore I modified the function we were given to convert the labels to teh ones required by the BERT model

In [7]:
def hierarchical_to_ner(tags):
    """
    Combine the EBM-NLP hierarchical labels (0–7) with BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B-1", "I-1", "B-2", "I-2"...).
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B-" + str(t))
            else:
                bio.append("I-" + str(t))
        prev = t
        

    return bio

def convert_all_labels_to_ner(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_ner(doc_labels)
    return labels

In [8]:
interventions_ner_labels = convert_all_labels_to_ner(interventions_labels)
test_interventions_ner_labels = convert_all_labels_to_ner(test_interventions_labels)

all_labels_updated = chain(*interventions_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_updated)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'B-7' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5'
 'I-6' 'I-7' 'O']


When using the huggingface API we need to create to dicts, to allow the model to convert the labels in to numbers and vice versa. So next we define the mappings from id to label and label to id

In [9]:
id2label_i = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
    9: "B-5",
    10: "I-5",
    11: "B-6",
    12: "I-6",
    13: "B-7",
    14: "I-7",
}

label2id_i = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
    "B-5": 9,
    "I-5": 10,
    "B-6": 11,
    "I-6": 12,
    "B-7": 13,
    "I-7": 14,
}

Then we convert the ner tags to their respective ids

In [10]:
def label_to_ids(labels_list, labels2ids):
    ids_list = []
    for label_list in labels_list:
        ids = []
        for label in label_list:
            ids.append(labels2ids[label])
        ids_list.append(ids)
    return ids_list

In [11]:
interventions_ner_ids = label_to_ids(interventions_ner_labels, label2id_i)
test_interventions_ner_ids = label_to_ids(test_interventions_ner_labels, label2id_i)
print(interventions_ner_ids[0])
print(test_interventions_ner_ids[0])

[0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 5, 6, 6, 6, 0, 5, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Now since we are using a pretrained model we can use the pretrained tokenizer to tokenize our input words. However we need to be careful when doing this as some words may be broken up into several tokens resulting in the tokens and labels being misaligned. To solve this issue we can use the Huggingface inbuilt tokenizer field `batch_id` which keeps track of multiple tokens belonging to the same word. Here we give each additional part of each token the number "-100". This we be used later to allow the model to ignore these extra parts (Note: after testing the model while the accuracy was high, over 95%, the other metrics (precision, recall and f1-score) were all around 40% signalling that the data is heavily skewed towards negative results and the model is over predicting negative outputs. After changing this function to include the extra parts of each split token the 3 metrics rose to over 55% for interventions) 

In [12]:
def tokenize_and_align_labels(inputs):
    tokenized_inputs = tokenizer(inputs["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(inputs["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i) # This gives the same word id for words that have been split up
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx: # Only label the first token of a given word
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
                #label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

We then need to convert the data into an objects that can be used by the API: The huggingface Dataset and DataDict (The DataDict isn't technically necessary but it can help keep all the data together so it is very convenient. Also most of the datasets available on huggingfaces are in the format of a DataDict)

In [13]:
from datasets import Dataset, DatasetDict

interventions_initial_train_ds = Dataset.from_dict({
    "tokens": interventions_tokens,
    "ner_tags": interventions_ner_ids,
})

test_interventions_initial_train_ds = Dataset.from_dict({
    "tokens": test_interventions_tokens,
    "ner_tags": test_interventions_ner_ids,
})

In [14]:
ds_dict_i = DatasetDict({
    "train": interventions_initial_train_ds,
    "test": test_interventions_initial_train_ds,
})

Now to use the above function we first need to initialize our tokenizer.  
Our first test will be the smallest BERT model: [DistilBERT (base-uncased)](https://huggingface.co/distilbert/distilbert-base-uncased)

In [15]:
from transformers import AutoTokenizer, DistilBertForTokenClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

Also using the Dataset datatype allows us to use its inbuilt map function which updates the data more efficiently

In [16]:
interventions_ds = ds_dict_i.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4746 [00:00<?, ? examples/s]

Map:   0%|          | 0/187 [00:00<?, ? examples/s]

Now that the data has been tokenized we still need to worry about alignment between data samples. As not all documents are the same length most documents will need to be either padded or truncated. Initially I wrote a function to carry this out but then I found that the API can carry this out using a DataCollator which was much more straightforward

In [17]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [18]:
import evaluate

seqeval = evaluate.load("seqeval")

In [19]:
import numpy as np

label_list_i = list(label2id_i.keys())

def compute_metrics_i(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_i[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)                   # or ignore special tokens added by tokenizer
    ]

    true_labels = [
        [label_list_i[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [20]:
model_i = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 15, id2label=id2label_i, label2id=label2id_i)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
from transformers import TrainingArguments, Trainer

In [22]:
training_args_i = TrainingArguments(
    output_dir = "interventions_classification_model",
    learning_rate = 1e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
   # optim="apollo_adamw",
   # optim_target_modules=[r".*.attention.*", r".*.ffn.*"],
    num_train_epochs = 5,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

In [23]:
trainer_i = Trainer(
    model = model_i,
    args = training_args_i,
    train_dataset = interventions_ds["train"],
    eval_dataset = interventions_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_i,
)

In [24]:
trainer_i.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.191883,0.462521,0.368385,0.410121,0.954177
2,0.305523,0.188270,0.367144,0.400271,0.382993,0.945768
3,0.305523,0.171040,0.415301,0.360923,0.386207,0.949952
4,0.187730,0.169538,0.400786,0.415197,0.407864,0.948497
5,0.187730,0.159479,0.437826,0.398915,0.417465,0.952479
6,0.171717,0.161855,0.422814,0.419946,0.421375,0.949023
7,0.157749,0.151257,0.452314,0.437585,0.444828,0.953934
8,0.157749,0.156653,0.441094,0.426730,0.433793,0.950276
9,0.148100,0.160369,0.419481,0.438263,0.428666,0.947224
10,0.148100,0.157190,0.436364,0.423338,0.429752,0.950054


/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2970, training_loss=0.18608428492690576, metrics={'train_runtime': 556.1728, 'train_samples_per_second': 85.333, 'train_steps_per_second': 5.34, 'total_flos': 6173046340093620.0, 'train_loss': 0.18608428492690576, 'epoch': 10.0})

Whilst the accuracy score is high, the low f1 score indicates that the model is correctly predicting the negative case which makes up most of the data, but not the positive cases that we want to predict. 

In [25]:
trainer_i.save_model("model/interventions_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Participants
Now to repeat the process with participants

In [26]:
doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

print(f"Number of documents in train split for participants: {len(doc_ids_p)}")
print(f"Number of documents in test split for participants: {len(test_doc_ids_p)}")

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)

Number of documents in train split for participants: 4609
Number of documents in test split for participants: 189


In [27]:
participants_labels = load_labels(doc_ids_p, "participants", split="train")
print(f"Length of participants_labels: {len(participants_labels)}")

test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")
print(f"Length of test_participants_labels: {len(test_participants_labels)}")

participants_ner_labels = convert_all_labels_to_ner(participants_labels)
test_participants_ner_labels = convert_all_labels_to_ner(test_participants_labels)

Length of participants_labels: 4609
Length of test_participants_labels: 189


In [28]:
all_labels_p = chain(*participants_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_p)))

['B-1' 'B-2' 'B-3' 'B-4' 'I-1' 'I-2' 'I-3' 'I-4' 'O']


This gives us the labels for the participants

In [29]:
id2label_p = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
}

label2id_p = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
}

In [30]:
participants_ner_ids = label_to_ids(participants_ner_labels, label2id_p)
test_participants_ner_ids = label_to_ids(test_participants_ner_labels, label2id_p)

In [31]:
participants_initial_train_ds = Dataset.from_dict({
    "tokens": participants_tokens,
    "ner_tags": participants_ner_ids,
})

test_participants_initial_train_ds = Dataset.from_dict({
    "tokens": test_participants_tokens,
    "ner_tags": test_participants_ner_ids,
})

In [32]:
ds_dict_p = DatasetDict({
    "train": participants_initial_train_ds,
    "test": test_participants_initial_train_ds,
})

In [33]:
participants_ds = ds_dict_p.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4609 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

In [34]:
label_list_p = list(label2id_p.keys())

def compute_metrics_p(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_p[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list_p[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [35]:
model_p = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 9, id2label=id2label_p, label2id=label2id_p)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
training_args_p = TrainingArguments(
    output_dir = "participants_classification_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 6,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_p = Trainer(
    model = model_p,
    args = training_args_p,
    train_dataset = participants_ds["train"],
    eval_dataset = participants_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_p,
)

In [37]:
trainer_p.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.168513,0.347349,0.222743,0.271429,0.943514
2,0.156896,0.137814,0.349650,0.293083,0.318878,0.948119
3,0.156896,0.153118,0.354467,0.288394,0.318035,0.946349
4,0.104136,0.148719,0.340127,0.313013,0.326007,0.948018
5,0.104136,0.150656,0.349105,0.320047,0.333945,0.949345
6,0.090393,0.143123,0.324945,0.348183,0.336163,0.950110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1734, training_loss=0.11291989433174178, metrics={'train_runtime': 323.3736, 'train_samples_per_second': 85.517, 'train_steps_per_second': 5.362, 'total_flos': 3594886826630778.0, 'train_loss': 0.11291989433174178, 'epoch': 6.0})

In [38]:
trainer_p.save_model("model/participants_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Outcomes

In [39]:
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print(f"Number of documents in train split for outcomes: {len(doc_ids_o)}")
print(f"Number of documents in test split for outcomes: {len(test_doc_ids_o)}")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

Number of documents in train split for outcomes: 4681
Number of documents in test split for outcomes: 190


In [40]:
outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
print(f"Length of outcomes_labels: {len(outcomes_labels)}")

test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")
print(f"Length of test_outcomes_labels: {len(test_outcomes_labels)}")

outcomes_ner_labels = convert_all_labels_to_ner(outcomes_labels)
test_outcomes_ner_labels = convert_all_labels_to_ner(test_outcomes_labels)

Length of outcomes_labels: 4681
Length of test_outcomes_labels: 190


In [41]:
all_labels_o = chain(*outcomes_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_o)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5' 'I-6'
 'O']


In [42]:
id2label_o = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
    9: "B-5",
    10: "I-5",
    11: "B-6",
    12: "I-6",
}

label2id_o = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
    "B-5": 9,
    "I-5": 10,
    "B-6": 11,
    "I-6": 12,
}

In [43]:
outcomes_ner_ids = label_to_ids(outcomes_ner_labels, label2id_o)
test_outcomes_ner_ids = label_to_ids(test_outcomes_ner_labels, label2id_o)

In [44]:
outcomes_initial_train_ds = Dataset.from_dict({
    "tokens": outcomes_tokens,
    "ner_tags": outcomes_ner_ids,
})

test_outcomes_initial_train_ds = Dataset.from_dict({
    "tokens": test_outcomes_tokens,
    "ner_tags": test_outcomes_ner_ids,
})

In [45]:
ds_dict_o = DatasetDict({
    "train": outcomes_initial_train_ds,
    "test": test_outcomes_initial_train_ds,
})

In [46]:
outcomes_ds = ds_dict_o.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4681 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

In [47]:
label_list_o = list(label2id_o.keys())

def compute_metrics_o(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_o[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list_o[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [48]:
model_o = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 13, id2label=id2label_o, label2id=label2id_o)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [49]:
training_args_o = TrainingArguments(
    output_dir = "outcomes_classification_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 4,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_o = Trainer(
    model = model_o,
    args = training_args_o,
    train_dataset = outcomes_ds["train"],
    eval_dataset = outcomes_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_o,
)

In [50]:
trainer_o.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.220561,0.204255,0.205274,0.204764,0.926891
2,0.323673,0.204129,0.241701,0.295795,0.266026,0.929864
3,0.323673,0.188436,0.301872,0.275837,0.288268,0.933677
4,0.225372,0.188058,0.289402,0.303635,0.296348,0.933297


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1172, training_loss=0.2649984750324549, metrics={'train_runtime': 219.3223, 'train_samples_per_second': 85.372, 'train_steps_per_second': 5.344, 'total_flos': 2433825492184440.0, 'train_loss': 0.2649984750324549, 'epoch': 4.0})

In [51]:
trainer_o.save_model("model/outcomes_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

We can see the same pattern repeated throughout, high accuracy (over 92%) but low on all other scores, indicating that each model is good at predicting negative results it is not good at predicting positive ones. In order to improve this situation we can try a different model, like one of the larger BERT models e.g. [bert-based-uncased](https://huggingface.co/google-bert/bert-base-uncased)

In [52]:
from transformers import BertForTokenClassification, BertConfig

bert_model_checkpoint = "bert-base-cased"

# 1. Load config and update max_position_embeddings
config = BertConfig.from_pretrained("bert-base-cased", num_labels=15, id2label=id2label_i, label2id=label2id_i)
config.max_position_embeddings = 1024 

# 2. Initialize model with new config
bert_i_model = BertForTokenClassification(config)

previous_weights_model = BertForTokenClassification.from_pretrained("bert-base-cased")
with torch.no_grad():
    bert_i_model.bert.embeddings.position_embeddings.weight[:512, :] = previous_weights_model.bert.embeddings.position_embeddings.weight


bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_checkpoint)

bert_data_collator = DataCollatorForTokenClassification(tokenizer=bert_tokenizer)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [53]:
def tokenize_and_align_labels2(inputs):
    tokenized_inputs = bert_tokenizer(inputs["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(inputs["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i) # This gives the same word id for words that have been split up
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            #elif word_idx != previous_word_idx: # Only label the first token of a given word
            #    label_ids.append(label[word_idx])
            else:
            #    label_ids.append(-100)
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [54]:
bert_interventions_ds = ds_dict_i.map(tokenize_and_align_labels2, batched=True)

Map:   0%|          | 0/4746 [00:00<?, ? examples/s]

Map:   0%|          | 0/187 [00:00<?, ? examples/s]

In [55]:
bert_training_args_i = TrainingArguments(
    output_dir = "bert_interventions_classification_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 4,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

bert_trainer_i = Trainer(
    model = bert_i_model,
    args = bert_training_args_i,
    train_dataset = bert_interventions_ds["train"],
    eval_dataset = bert_interventions_ds["test"],
    processing_class = bert_tokenizer,
    data_collator = bert_data_collator,
    compute_metrics = compute_metrics_i,
)

In [56]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [57]:
bert_trainer_i.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.410447,0.665138,0.063849,0.116513,0.906012
2,0.479815,0.398936,0.514595,0.314399,0.390324,0.910147
3,0.479815,0.373356,0.599595,0.260458,0.363162,0.912740
4,0.413943,0.368935,0.566270,0.293483,0.386601,0.912624


/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1188, training_loss=0.4398055349536215, metrics={'train_runtime': 434.8073, 'train_samples_per_second': 43.661, 'train_steps_per_second': 2.732, 'total_flos': 4957128897870240.0, 'train_loss': 0.4398055349536215, 'epoch': 4.0})

In [58]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [59]:
bert_trainer_i.save_model("model/bert_interventions_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]